# Swarm Renormalization Field Learning (SRFL)
## Interactive Walkthrough

**Author:** Bishal Neupane — Astronomy Squad of Koshi  
**Repository:** [github.com/cosmobishal/SRFL](https://github.com/cosmobishal/SRFL)

This notebook demonstrates the full SRFL pipeline on two canonical targets:
1. **Heaviside step** `H(x)` — a jump discontinuity
2. **Oscillatory singularity** `x·sin(1/x)` — an essential singularity at `x=0`

---

In [ ]:
import sys
sys.path.insert(0, '../src')  # run from notebooks/ directory

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

from srfl import (
    SRFLKernel, SRFLField, Swarm,
    ActionFunctional, ScaleProjection, DefectAlgebra,
    StepDefect, OscillatoryDefect
)
print('SRFL imported successfully.')

## 1. Setup: Grid and Target Functions

In [ ]:
# Spatial grid
N         = 512
x         = np.linspace(-np.pi, np.pi, N)
dx        = float(x[1] - x[0])

# Scale schedule: λ flows from coarse (1.0) to fine (0.015)
S         = 70
lam_sched = np.logspace(0, -1.85, S)

# Target functions
target_step = np.where(x >= 0, 1.0, 0.0).astype(float)
target_osc  = np.where(x != 0, x * np.sin(1.0 / np.where(x != 0, x, 1.0)), 0.0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(x, target_step, 'k-', lw=2)
axes[0].set_title('Target A: Heaviside $H(x)$'); axes[0].grid(True, alpha=0.3)
axes[1].plot(x, target_osc, 'purple', lw=1.5)
axes[1].set_xlim(-2, 2); axes[1].set_ylim(-1.5, 1.5)
axes[1].set_title(r'Target B: $x \cdot \sin(1/x)$'); axes[1].grid(True, alpha=0.3)
plt.tight_layout()

## 2. Non-Local Gaussian Kernel `K(x, x', λ)`

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for lam, color in zip([1.0, 0.5, 0.2, 0.05], ['navy','royalblue','steelblue','skyblue']):
    K = SRFLKernel(x, lam)
    k_profile = np.exp(-x**2 / (2 * lam**2))
    k_profile /= k_profile.max()
    ax.plot(x, k_profile, color=color, lw=2, label=f'λ={lam}  FWHM={K.fwhm():.3f}')
ax.set_title('Kernel $K(x, x\'=0, λ)$ for varying $λ$')
ax.set_xlabel('$x$'); ax.set_ylabel('$K$ (normalised)')
ax.legend(); ax.grid(True, alpha=0.3); ax.set_xlim(-2, 2)

## 3. SRFL Field Evolution — Step Function

In [ ]:
engine_step = SRFLField(x, target_step, lam_sched, dt=0.28)
fields_step, errors_step = engine_step.run(verbose=True)
rate_step = engine_step.convergence_rate(errors_step)
print(f'\nConvergence rate r ≈ {rate_step:.3f}  (L² error ~ λ^r)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sel  = np.linspace(0, S - 1, 8, dtype=int)
cmap = plt.cm.viridis
ax = axes[0]
for k, idx in enumerate(sel):
    c = k / (len(sel) - 1)
    ax.plot(x, fields_step[idx], color=cmap(0.9 - 0.75*c),
            lw=1.8, label=f'λ={lam_sched[idx]:.3f}', alpha=0.9)
ax.plot(x, target_step, 'r--', lw=1.5, alpha=0.6, label='H(x)')
ax.set_title('Field Evolution Φ(x, λ)'); ax.set_xlabel('x'); ax.set_ylabel('Φ')
ax.legend(fontsize=7, ncol=2); ax.grid(True, alpha=0.2)

ax = axes[1]
ax.semilogy(lam_sched, errors_step, 'b-o', ms=3, lw=2)
ax.invert_xaxis()
ax.set_xlabel('λ  (coarse → fine)'); ax.set_ylabel('L² error')
ax.set_title('Convergence ‖Φ − H‖'); ax.grid(True, which='both', alpha=0.25)
plt.tight_layout()

## 4. Swarm Lifecycle — Spawn / Merge / Annihilate

In [ ]:
swarm = Swarm(x, n_init=14)
for k, (phi, lam) in enumerate(zip(fields_step[1:], lam_sched[1:]), 1):
    swarm.step(phi, target_step, lam, k)

print('Swarm event summary:', swarm.event_summary())
print(f'Final agent count: {swarm.count()}')

fig, ax = plt.subplots(figsize=(12, 6))
n_h = len(swarm.history)
for s, pos in enumerate(swarm.history):
    c = s / n_h
    ax.scatter(pos, np.full(len(pos), s), s=14, color=cmap(1 - c), alpha=0.65)
for (s, etype, ex) in swarm.events:
    m = {'spawn':'^', 'merge':'D', 'annihilate':'x'}[etype]
    c = {'spawn':'lime', 'merge':'gold', 'annihilate':'red'}[etype]
    ax.scatter(ex, s, marker=m, s=80, color=c, zorder=5)
ax.set_xlabel('x'); ax.set_ylabel('Scale step s')
ax.set_title('SRFL Swarm Dynamics — Step Function')
ax.grid(True, alpha=0.2)

## 5. Action Functional 𝒜 Diagnostics

In [ ]:
action = ActionFunctional(x, lam_sched, target_step)
A = action.total(fields_step)

print('Action functional components:')
for k, v in A.items():
    print(f'  𝒜_{k:<12} = {v:.6f}')

## 6. Scale Projection — Semigroup Verification

In [ ]:
proj = ScaleProjection(x)
phi_test = np.sin(x) + 0.3 * np.cos(3*x)
err, ok = proj.verify_semigroup(phi_test, lam1=0.2, lam2=0.5, lam3=0.8)
print(f'Semigroup L² error: {err:.2e}  (ok={ok})')

sp_err = proj.consistency_profile(fields_step, lam_sched, stride=5)
fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(np.arange(len(sp_err))[5:], sp_err[5:] + 1e-12, 'g-', lw=2)
ax.set_xlabel('Scale step s'); ax.set_ylabel('Consistency error')
ax.set_title('Scale Consistency ‖Φ(λₖ) − Π·Φ(λₖ₋₁)‖')
ax.grid(True, which='both', alpha=0.25)

## 7. Defect Detection in Final Field

In [ ]:
alg     = DefectAlgebra(x)
defects = alg.detect_from_curvature(x, fields_step[-1], dx)
print(f'Detected {len(defects)} defect(s) in final field:')
for d in defects:
    print(f'  {d}')

# Visualise curvature and detected defect regions
phi_final = fields_step[-1]
d2 = np.gradient(np.gradient(phi_final, dx), dx)
thr = np.percentile(np.abs(d2), 90)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(x, phi_final, 'b-', lw=2, label='Φ (final)', zorder=3)
ax.plot(x, target_step, 'r--', lw=1.5, alpha=0.6, label='H(x)')
ax.fill_between(x, -0.3, 1.3, where=np.abs(d2) > thr,
                color='crimson', alpha=0.2, label='Defect zone')
ax.set_xlim(-1.5, 1.5); ax.set_ylim(-0.3, 1.3)
ax.set_title('Final Field with Detected Defect Zones')
ax.set_xlabel('x'); ax.legend(); ax.grid(True, alpha=0.2)

## 8. SRFL CLI Usage

From the repository root, the CLI can be invoked as:

```bash
# After pip install -e .
srfl-run --target step   --n 512 --steps 70 --outdir figures
srfl-run --target osc    --n 512 --steps 70 --outdir figures
srfl-run --target both   --outdir figures --verbose
srfl-run --list-targets
```

Or directly:
```bash
python -m srfl.cli --target step
```